In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

brightness = ".,-+><[]"

In [ ]:
img = Image.open("turing.jpg")
np.array(img).shape

In [ ]:
pool = nn.MaxPool2d(4)
img_t = torch.from_numpy(np.array(img))
img_t.unsqueeze_(0)
compress_4x = pool(img_t).squeeze()

In [ ]:
plt.hist(compress_4x.flatten(), bins=8)

In [ ]:
compressed = torch.floor(compress_4x / 256 * 8).to(torch.int8).tolist()
result = ""
for row in compressed:
    for cell in row:
        result += brightness[cell]
    result += "\n"

In [ ]:
plt.imshow(compressed)

In [ ]:
with open("turing.txt", "w") as f:
    f.write(result)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="../tmp",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="../tmp",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
printable_ascii = list(map(chr, range(32, 127)))

In [ ]:
print(printable_ascii)

In [ ]:
size = 100
font_path = "./Retro Gaming.ttf"
font = ImageFont.truetype(font_path, size)

def get_brightness(text):
    img = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(img)
    draw.text((0, 0), text, font=font, fill=255, anchor="lt")
    brigtness = np.array(img).sum() / 256 / size**2
    return brigtness

In [ ]:
text = "@"
img = Image.new("L", (size, size), 0)
draw = ImageDraw.Draw(img)
draw.text((0, 0), text, font=font, fill=255, anchor="lt")
brightness = np.array(img).sum() / 256 / (size**2)
print(np.array(img).sum()/255)

In [ ]:
get_brightness("0")

In [ ]:
get_brightness("#")

In [ ]:
ascii_bright_list = list(map(get_brightness, printable_ascii))
plt.hist(ascii_bright_list, range=(0,1), bins=100)

In [ ]:
ascii_bright_map = {k:get_brightness(k) for k in printable_ascii}
# ascii_from_bright = list(sorted(printable_ascii, key=get_brightness, reverse=True))
e = max(ascii_bright_list)
prev = -1
count = 0

selected = []
for (k, v) in sorted(ascii_bright_map.items(), key=lambda x: x[1]): # start with 16 chan
    # print(k, end=" ")
    if v-prev > (2**(count/85)-1.07):
        count += 1
        selected.append(k)
        prev = v


print(selected, len(selected))

In [ ]:
plt.plot(range(len(selected)), list(map(ascii_bright_map.__getitem__, selected)))

In [ ]:
# .-':_,^=;><+!rc*/z?sLTv)J7(|Fi{C}fI31tlu[neoZ5Yxjya]2ESwqkP6h9d4VpOGbUAKXHm8RD#$Bg0MNWQ%&@
def quantize_img(tensor_img, kernel_size=2):
    return F.max_pool2d(torch.floor(tensor_img*15.9), kernel_size=kernel_size).to(torch.long)
    
def to_ascii_img(quantized_img):
    result = []
    for row in quantized_img.tolist()[0]:
        ascii_row = ""
        for cell in row:
            ascii_row += selected[int(cell)]
        result.append(f"{ascii_row:<14}")
    return result

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),   # converts PIL image → [0,1] tensor
    quantize_img
])

train_dataset = datasets.FashionMNIST(
    root="../tmp",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="../tmp",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
print("\n".join(to_ascii_img(train_dataset[0][0])))

In [ ]:
print("\n".join(to_ascii_img(train_dataset[1][0])))

In [ ]:
train_dataset[1][0].size()

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

imgs, _ = next(iter(train_loader))
plt.hist(imgs.flatten(), density=True)

In [ ]:
classes = train_dataset.classes

def fashion_show(dataset, name):
    width = 14
    hpadding = 4
    hspace = (width+hpadding)
    result = [""]*hspace*32
    for i in range(32):
        for j in range(5):
            tensor_img, label = dataset[5*i+j]
            ascii_img = to_ascii_img(tensor_img)
            for ii, row in enumerate(ascii_img):
                result[2+ii+i*hspace] += row + " "*4
            result[i*hspace] += f"{classes[label]:<18}"
    with open(f"fashion_show_{name}.txt", "w") as f:
        f.write("\n".join(result))

fashion_show(train_dataset, "train")

In [ ]:
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).bool())

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)                       # (B,T,3C)
        q, k, v = qkv.chunk(3, dim=-1)          # each (B,T,C)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B,H,T,D)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)     # (B,H,T,T)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)

        out = att @ v                                                  # (B,H,T,D)
        out = out.transpose(1, 2).contiguous().view(B, T, C)           # (B,T,C)
        return self.proj(out)

class MLP(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # residual
        x = x + self.mlp(self.ln2(x))   # residual
        return x

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, block_size=64, d_model=128, n_heads=4, n_layer=1):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok(idx) + self.pos(pos)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B,T,V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
cfg = {"cmap": "gray", "vmin": 0, "vmax": 15}
plt.imshow(train_dataset[0][0].squeeze(), **cfg)

In [ ]:
vocab = selected + train_dataset.classes + ["START"]
START = len(vocab)-1
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for ch,i in stoi.items()}

In [ ]:
tensor_img, label = train_dataset[0]
Y = torch.roll(tensor_img.flatten(), 1)
Y[0] = START
X = torch.roll(Y, 1)
X[0] = label

In [ ]:
print(len(vocab))

In [ ]:
def collate_fn(batch):
    xs, labels = zip(*batch)
    xs = torch.stack(xs)
    labels = torch.tensor(labels)
    ys = torch.flatten(xs, 1, -1)
    xs = torch.roll(ys, shifts=1, dims=1)
    xs[:, 0] = labels+len(selected)
    return xs, ys

train_loader = DataLoader(train_dataset, batch_size=128, collate_fn=collate_fn)

In [ ]:
len(train_dataset)

In [ ]:
x, y = next(iter(train_loader))
print(x.shape)
x

In [ ]:
device = "cpu"
block_size = 196
model = TinyTransformer(vocab_size=len(vocab), block_size=block_size, d_model=64, n_heads=4, n_layer=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

sum([p.nelement() for (n, p) in model.named_parameters()])

In [ ]:
losses = []

loop = iter(train_loader)
for step in range(10000):
    try:
        x, y = next(loop)
    except StopIteration:
        loop = iter(train_loader)   
        step -= 1
        continue
    x, y = x.to(device), y.to(device)
    _, loss = model(x, y)
    losses.append(loss.item())
    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 250 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

In [ ]:
torch.save(model.state_dict(), "pixel_1layer_10000.pt")

In [ ]:
plt.plot(losses)

In [ ]:
print(train_dataset.classes)
print(itos)

In [ ]:
@torch.no_grad()
def sample(model, start, steps=200, temperature=1.0):
    model.eval()
    idx = start
    for _ in range(steps):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

# start = torch.tensor([[20]], device=device)
# out = sample(model, start, steps=200)

# def parse_img(out):
#     s = "".join([itos[tok] for tok in out[1:]])
#     result = []
#     for i in range(14):
#         result.append(s[i*14:(i+1)*14])
#     return result

# print("\n".join(parse_img(out[0].tolist())))

In [ ]:
# 32, 5
start = torch.randint(0, len(train_dataset.classes), (160, 1), device=device) + len(selected)
out = sample(model, start, steps=196)
imgs = out[:, 1:].view(-1, 1, 14, 14).clip(0, len(selected)-1)
labels = out[:, 0] - len(selected)
class PseudoDataset():
    def __init__(self, tensor_imgs, labels):
        self.tensor_imgs = tensor_imgs
        self.labels = labels
    def __getitem__(self, i):
        return (self.tensor_imgs[i], self.labels[i])

pseudo_dataset = PseudoDataset(imgs, labels)
fashion_show(pseudo_dataset, "1layer_10k")

In [ ]:
pseudo_dataset[0]

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

# path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-dataset")
handle = "paramaggarwal/fashion-product-images-small"
path = kagglehub.dataset_download(handle)
df = pd.read_csv(path+"/styles.csv", on_bad_lines='skip')

df["productDisplayName"]

In [ ]:
from functools import partial

transform = transforms.Compose([
    transforms.ToTensor(),
    partial(quantize_img, kernel_size=10)
])
to_pil = transforms.ToPILImage()

tensor_img = transforms.ToTensor()(Image.open("snu_jacket.png"))
transformed_img = quantize_img(tensor_img, kernel_size=10)
grayscale_img = transformed_img[:-1].sum(0, keepdims=True) / 3
contrast_img = 15-grayscale_img
# cfg = {"cmap": "gray", "vmin":0, "vmax": 1}

with open("snu_jacket.txt", "w") as f:
    f.write("\n".join(to_ascii_img(contrast_img)))

to_pil(contrast_img/15).show()

In [ ]:
to_pil(tensor_img)

In [ ]:
edge_ascii = "/\\_|"

horizontal = torch.tensor([[
    [0., 0, 0],
    [1, 1, 1],
    [-1, -1, -1]
]])
vertical = torch.tensor([[
    [0., 1, -1],
    [0, 1, -1],
    [0, 1, -1]
]])
right_upward = torch.tensor([[
    [0., 1, 0],
    [1, -1, 0],
    [-1, 0, 0]
]])
left_upward = torch.tensor([[
    [0., 1, 0],
    [0, -1, 1],
    [0, 0, -1]
]])

In [ ]:
import cv2

img = cv2.imread("snu_jacket.png")
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
s = hsv[:,:,1]

mask = s > 30

In [ ]:
plt.imshow(img.mean(-1)*mask, cmap="gray", vmin=0, vmax=1)

In [ ]:
to_pil(tensor_img*mask)

In [ ]:
tensor_img.shape


In [ ]:
threshold = 0.6
def filter_transform(tensor_img, filter, threshold=0.55):
    filtered = F.conv2d(tensor_img, filter.unsqueeze(0), padding=1).abs() > threshold
    return filtered*1.0

small = F.max_pool2d(tensor_img.mean(0, keepdim=True), 12)
small_mask = F.max_pool2d(torch.tensor(mask*1.0).unsqueeze(0), 12)
# Generate the four images
# to_pil(small*small_mask).show()
img1 = to_pil(filter_transform(small, horizontal))
img2 = to_pil(filter_transform(small, vertical))
img3 = to_pil(filter_transform(small, right_upward, 0.4))
img4 = to_pil(filter_transform(small, left_upward, 0.4))

w, h = img1.size
grid = Image.new("RGB", (2 * w, 2 * h))
grid.paste(img1, (0, 0))
grid.paste(img2, (w, 0))
grid.paste(img3, (0, h))
grid.paste(img4, (w, h))
grid

In [ ]:
quantized_img = torch.floor((small * small_mask)*15.9)

with open("snu_jacket.txt", "w") as f:
    f.write("\n".join(to_ascii_img(quantized_img)))

In [ ]:
def to_ascii_img(quantized_img):
    result = []
    for row in quantized_img.tolist()[0]:
        ascii_row = ""
        for cell in row:
            ascii_row += selected[int(cell)]
        result.append(f"{ascii_row:<14}")
    return result

In [ ]:
filters = torch.randn(1, 1, 3, 3)
inputs = torch.randn(1, 5, 5)
F.conv2d(inputs, filters, padding=1)

In [ ]:
one_chan.unsqueeze(0).shape, horizontal.shape